# Prototype B (cinema): render the new interior frames int2_* on Colab
Run top to bottom on an L4. Frames are written straight to Google Drive (`MyDrive/homepage-v6-int2`), so after a disconnect just run all cells again: finished frames are skipped.

Renders 98 frames: the 86 odd frames int2_0001..int2_0171, plus the even approach frames int2_0000..int2_0022 (set `APPROACH_EVENS = '0'` below to skip those 12; see README.txt why). Even frames from int2_0024 on = shipped int_k, k = t/2 (not rendered here).

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/homepage-v6-int2'
for d in ('jpg', '1920', '1280'):
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
print(DRIVE, {d: len(os.listdir(f'{DRIVE}/{d}')) for d in ('jpg', '1920', '1280')})

In [ ]:
%%bash
# Blender 4.5.14 LTS (linux x64) + the X/GL libs it links against, once.
set -e
if [ ! -x /content/blender/blender ]; then
  apt-get -qq update && apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libxkbcommon0 libsm6 > /dev/null
  wget -q https://download.blender.org/release/Blender4.5/blender-4.5.14-linux-x64.tar.xz -O /content/blender.tar.xz
  mkdir -p /content/blender && tar -xf /content/blender.tar.xz -C /content/blender --strip-components=1
  rm /content/blender.tar.xz
fi
/content/blender/blender --version | head -1

In [ ]:
# Kit: from Drive (upload kit-b.zip to MyDrive/homepage-v6-int2/ by hand) or from KIT_URL once it is hosted.
import zipfile, subprocess
KIT_URL = 'https://github.com/mungvu2004/AppFront/raw/render-kit-b/kit-b.zip'
KIT = '/content/kit-b'
if not os.path.isdir(f'{KIT}/cinema/scripts'):
    z = f'{DRIVE}/kit-b.zip'
    if not os.path.exists(z):
        assert KIT_URL, 'put kit-b.zip in ' + DRIVE + ' or set KIT_URL'
        z = '/content/kit-b.zip'
        os.system(f'wget -q {KIT_URL} -O {z}')
    zipfile.ZipFile(z).extractall(KIT)
os.environ['KIT_ROOT'] = KIT
# rewrites the scripts' F:/ paths to /content/kit-b and downloads 218 Poly Haven CC0 files (486 MB) from dl.polyhaven.org
!python {KIT}/kit.py setup

In [ ]:
# Build the scene (same scripts that built cinema.blend on the PC): plant library, then the master scene.
import time
BL = '/content/blender/blender'
S = f'{KIT}/cinema/scripts'
for script, out in (('make_veg_lib.py', 'veg_lib.blend'), ('build_scene.py', 'cinema.blend')):
    if not os.path.exists(f'{KIT}/cinema/blend/{out}'):
        t0 = time.time()
        r = subprocess.run([BL, '-b', '-P', f'{S}/{script}'], capture_output=True, text=True)
        print(script, 'exit', r.returncode, f'{time.time() - t0:.0f} s')
        if r.returncode or not os.path.exists(f'{KIT}/cinema/blend/{out}'):
            print(r.stdout[-3000:], r.stderr[-3000:]); raise SystemExit('build failed')
print(os.listdir(f'{KIT}/cinema/blend'))

In [ ]:
# Render helper: one Blender process per chunk, frames written directly to Drive (render.py --list).
import sys
os.environ['APPROACH_EVENS'] = '1'  # '0' = odd frames only (86)
sys.path.insert(0, KIT)
import kit
from PIL import Image
SPP = 160  # same as the shipped int_ frames (adaptive + OIDN, settings in lib.setup_render)
def ok(path):
    try:
        Image.open(path).verify(); return True
    except Exception:
        return False
def todo():
    out = []
    for name, path, p in kit.frames():
        f = f'{DRIVE}/jpg/{name}.jpg'
        if os.path.exists(f) and not ok(f):
            os.remove(f)  # half-written by a dropped session
        if not os.path.exists(f):
            out.append((name, path, p))
    return out
def render(items):
    path = items[0][1]
    lst = ';'.join(f'{p!r}={DRIVE}/jpg/{n}.jpg' for n, _, p in items)
    cmd = [BL, '-b', f'{KIT}/cinema/blend/cinema.blend', '-P', f'{S}/render.py', '--', '--path', path, '--pass', 'beauty', '--samples', str(SPP), '--list', lst]
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if line.startswith('RENDER_S') or 'Error' in line or 'Traceback' in line or 'OPTIX' in line:
            print(line.rstrip(), flush=True)
    p.wait()
    print(f'{items[0][0]}..{items[-1][0]}: exit {p.returncode}, {time.time() - t0:.0f} s', flush=True)
print(len(todo()), 'frames to render')

In [ ]:
# Timing test: 1 frame (first OptiX run compiles kernels; scene load included).
t = todo()
if t:
    render(t[:1])

In [ ]:
# All remaining frames, chunks of 8 per camera path (re-run after a disconnect).
from itertools import groupby
for path, grp in groupby(todo(), key=lambda x: x[1]):
    grp = list(grp)
    for i in range(0, len(grp), 8):
        render(grp[i:i + 8])
print('left:', len(todo()))

In [ ]:
# WebP exactly like the shipped set (webp_combo.py): 1920 + 1280 LANCZOS, q76, method 6 -> Drive
done = 0
for f in sorted(os.listdir(f'{DRIVE}/jpg')):
    if not f.endswith('.jpg'):
        continue
    name = f[:-4] + '.webp'
    if os.path.exists(f'{DRIVE}/1280/{name}'):
        continue
    im = Image.open(f'{DRIVE}/jpg/{f}').convert('RGB')
    im.save(f'{DRIVE}/1920/{name}', quality=76, method=6)
    im.resize((1280, 720), Image.LANCZOS).save(f'{DRIVE}/1280/{name}', quality=76, method=6)
    done += 1
for w in ('1920', '1280'):
    fs = os.listdir(f'{DRIVE}/{w}')
    print(w, len(fs), 'frames', round(sum(os.path.getsize(f'{DRIVE}/{w}/{x}') for x in fs) / 1e6, 1), 'MB')

In [ ]:
# Zip the WebP sets (1920/int2_NNNN.webp, 1280/int2_NNNN.webp) + mapping note and download (the zip also stays on Drive).
import shutil
shutil.copy(f'{KIT}/README.txt', f'{DRIVE}/MAPPING.txt')
!cd {DRIVE} && rm -f int2-odd.zip && zip -q -r int2-odd.zip 1920 1280 MAPPING.txt && ls -la int2-odd.zip
from google.colab import files
files.download(f'{DRIVE}/int2-odd.zip')